In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI

# Cargar las variables de entorno desde .env
load_dotenv(override=True)

# Obtener la clave de API
api_key = os.getenv("OPENAI_API_KEY")
# Inicializar cliente de OpenAI
client = OpenAI(api_key=api_key)

# Hacer una solicitud
completion = client.chat.completions.create(
    model="o1-mini",
    messages=[
        {
            "role": "user",
            "content": "Write a one-sentence bedtime story about a unicorn."
        }
    ]
)

print(completion.choices[0].message.content)



Starting OpenAI API processing...
Loaded 113 problems from sample_selected.csv.

--- Processing: Model=o1-mini ---
Processing Problem 762...
Error: 'ChatCompletion' object is not subscriptable
Error processing Problem 762: No response from model
Processing Problem 1015...


KeyboardInterrupt: 

In [15]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

models = client.models.list()
for model in models.data:
    print(model.id)


gpt-4o-realtime-preview-2024-12-17
gpt-4o-audio-preview-2024-12-17
dall-e-3
dall-e-2
gpt-4o-audio-preview-2024-10-01
gpt-4o-mini-realtime-preview-2024-12-17
gpt-4o-mini-realtime-preview
gpt-4o-realtime-preview-2024-10-01
gpt-4o-transcribe
gpt-4o-mini-transcribe
gpt-4o-realtime-preview
babbage-002
gpt-4o-mini-tts
tts-1-hd-1106
text-embedding-3-large
gpt-4
text-embedding-ada-002
omni-moderation-latest
tts-1-hd
gpt-4o-mini-audio-preview
gpt-4o-audio-preview
o1-preview-2024-09-12
gpt-3.5-turbo-instruct-0914
gpt-4o-mini-search-preview
tts-1-1106
davinci-002
gpt-3.5-turbo-1106
gpt-4-turbo
gpt-4-0125-preview
gpt-3.5-turbo-instruct
gpt-3.5-turbo
gpt-4-turbo-preview
chatgpt-4o-latest
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-2024-11-20
whisper-1
gpt-3.5-turbo-0125
gpt-4o-2024-05-13
gpt-3.5-turbo-16k
gpt-4-turbo-2024-04-09
gpt-4-1106-preview
o1-preview
gpt-4-0613
gpt-4o-search-preview
gpt-4.5-preview
gpt-4.5-preview-2025-02-27
gpt-4o-search-preview-2025-03-11
tts-1
omni-moderation-2024-09-26


In [12]:
pip install --upgrade openai

   ---------------------------------------- 0.0/599.1 kB ? eta -:--:--
   ---------------------------------------- 599.1/599.1 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.66.0
    Uninstalling openai-1.66.0:
      Successfully uninstalled openai-1.66.0
Note: you may need to restart the kernel to use updated packages.


In [2]:

# Cargar las variables de entorno desde .env
load_dotenv(override=True)

# Obtener la clave de API e inicializar cliente de OpenAI
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# Modelos y parámetros
MODELS = ["o1-mini"]
TIMEOUT_EXEC = 300
INPUT_FILENAME = "sample_selected.csv"  # Usar el sample existente
OUTPUT_DIR = "outputs_openai"
RESULTS_FILE = "results_openai.csv"

def extract_code(response_text: str) -> str:
    if "```python" in response_text:
        return response_text.split("```python")[1].split("```")[0].strip()
    elif "```" in response_text:
        return response_text.split("```")[1].strip()
    return response_text.strip()

def measure_complexity(code: str) -> str:
    try:
        tree = ast.parse(code)
    except Exception:
        return "N/A"
    loops = sum(isinstance(node, (ast.For, ast.While)) for node in ast.walk(tree))
    conditionals = sum(isinstance(node, ast.If) for node in ast.walk(tree))
    return f"loops: {loops}, conditionals: {conditionals}"

def run_script(script_path: str, timeout: int):
    start_time = time.time()
    process = psutil.Popen(["python", script_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    max_mem = 0
    stdout, stderr = b"", b""
    while True:
        if process.poll() is not None:
            stdout, stderr = process.communicate()
            break
        if time.time() - start_time > timeout:
            process.kill()
            return "TIMEOUT", timeout, max_mem / (1024 * 1024)
        try:
            mem = process.memory_info().rss
            if mem > max_mem:
                max_mem = mem
        except psutil.NoSuchProcess:
            break
        time.sleep(0.1)
    exec_time = time.time() - start_time
    result = stdout.decode().strip() if stdout else stderr.decode().strip()
    return result, round(exec_time, 2), round(max_mem / (1024 * 1024), 2)

def call_model(model: str, prompt: str):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
        # Use attribute access for the usage object.
        text = response.choices[0].message.content
        usage = response.usage
        print(f"Response: {text}")
        return text, usage.prompt_tokens, usage.completion_tokens
    except Exception as e:
        print(f"Error in call_model for model {model}: {e}")
        return None, None, None

def main():
    print("Starting OpenAI API processing...")
    try:
        df = pd.read_csv(INPUT_FILENAME, encoding="utf-8")
    except Exception as e:
        print(f"Error reading file {INPUT_FILENAME}: {e}")
        return

    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    file_exists = os.path.exists(RESULTS_FILE)

    fieldnames = [
        "ID",
        "model",
        "code",
        "result",
        "true_count",
        "false_count",
        "execution_time",
        "memory_usage_MB",
        "algorithmic_complexity",
        "input_tokens",
        "output_tokens",
        "full_response"
    ]

    with open(RESULTS_FILE, "a", newline='', encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists or os.stat(RESULTS_FILE).st_size == 0:
            writer.writeheader()
        
        print(f"Loaded {len(df)} problems from {INPUT_FILENAME}.")
        for model in MODELS:
            print(f"\n--- Processing: Model={model} ---")
            for i, row in df.iterrows():
                print(f"Processing Problem {row['ID']}...")
                prompt = (
                    "Solve the following problem in Python. The solution should implement a function that, "
                    "given the specified input(s) in the problem, compares its output with the expected output and "
                    "runs the tests. The function should print 'True' for each test passed and 'False' for each test "
                    "failed, and finally print the number of correct tests over the total. "
                    "Provide only executable Python code.\n\n" + row["Description"]
                )

                # Initialize variables to use in error handling.
                response_text, in_tokens, out_tokens = None, None, None
                try:
                    response_text, in_tokens, out_tokens = call_model(model, prompt)
                    print(f"Response from model: {response_text}")
                    if response_text is None:
                        raise Exception("No response from model")
                    code = extract_code(response_text)
                    file_path = f"{OUTPUT_DIR}/output_{row['ID']}.py"
                    try:
                        with open(file_path, "w", encoding="utf-8") as f_out:
                            f_out.write(code)
                    except Exception as fe:
                        raise Exception(f"Error writing file {file_path}: {fe}")

                    result, exec_time, mem_usage = run_script(file_path, TIMEOUT_EXEC)
                    complexity = measure_complexity(code)
                    
                    true_count = result.count("True") if isinstance(result, str) else 0
                    false_count = result.count("False") if isinstance(result, str) else 0

                    result_row = {
                        "ID": row["ID"],
                        "model": model,
                        "code": code,
                        "result": result,
                        "true_count": true_count,
                        "false_count": false_count,
                        "execution_time": exec_time,
                        "memory_usage_MB": mem_usage,
                        "algorithmic_complexity": complexity,
                        "input_tokens": in_tokens or "N/A",
                        "output_tokens": out_tokens or "N/A",
                        "full_response": response_text
                    }
                    writer.writerow(result_row)
                    f.flush()  # Force flush to disk.
                    print(f"Processed Problem {row['ID']} successfully.")
                except Exception as e:
                    print(f"Error processing Problem {row['ID']}: {e}")
                    error_row = {
                        "ID": row["ID"],
                        "model": model,
                        "code": "",
                        "result": str(e),
                        "true_count": "ERROR",
                        "false_count": "ERROR",
                        "execution_time": "ERROR",
                        "memory_usage_MB": "ERROR",
                        "algorithmic_complexity": "ERROR",
                        "input_tokens": "N/A",
                        "output_tokens": "N/A",
                        "full_response": response_text if response_text is not None else "N/A"
                    }
                    writer.writerow(error_row)
                    f.flush()
                time.sleep(1)  # Pause to avoid rate limit issues

    print(f"Results saved incrementally to {RESULTS_FILE}")

if __name__ == "__main__":
    main()

Starting OpenAI API processing...
Loaded 113 problems from sample_selected.csv.

--- Processing: Model=o1-mini ---
Processing Problem 762...
Response: ```python
def count_prime_set_bits(L, R):
    primes = {2, 3, 5, 7, 11, 13, 17, 19}
    count = 0
    for num in range(L, R + 1):
        set_bits = bin(num).count('1')
        if set_bits in primes:
            count += 1
    return count

def run_tests():
    test_cases = [
        {'L': 6, 'R': 10, 'expected': 4},
        {'L': 10, 'R': 15, 'expected': 5},
        # You can add more test cases here
    ]
    
    correct = 0
    total = len(test_cases)
    
    for test in test_cases:
        result = count_prime_set_bits(test['L'], test['R'])
        if result == test['expected']:
            print('True')
            correct += 1
        else:
            print('False')
    
    print(f"{correct} / {total}")

run_tests()
```
Response from model: ```python
def count_prime_set_bits(L, R):
    primes = {2, 3, 5, 7, 11, 13, 17, 19}
    